# Paso 1: predicción del siguiente token sobre el corpus chileno (Gemma 4 E4B base)

Ajuste de dominio: el modelo base sigue pre-entrenándose (pérdida de lenguaje causal) sobre texto chileno para aprender cómo se expresa el chileno. El resultado es un **adaptador LoRA** que `02_paso2_traduccion.ipynb` fusiona en el modelo antes del ajuste a la traducción (condición `paso1+2`).

**Qué se entrena y qué queda congelado**
- LoRA **solo en la red feed-forward por posición** de cada capa del decodificador (`gate_proj`, `up_proj`, `down_proj`). Las FFN funcionan como memorias clave-valor del conocimiento léxico (Geva et al., 2021), que es lo que queremos inyectar, y no tocar la atención reduce el olvido de lo que el modelo ya sabe.
- **Congelados**: atención, embeddings de entrada y salida, embeddings por capa (PLE) de Gemma 4, normalizaciones y las torres de visión y audio. El tokenizer de Gemma ya segmenta bien el chileno, y entrenar los embeddings (~3.400 M parámetros en E4B) sería caro y riesgoso.
- Para la ablación "FFN + atención", cambia `MODULOS_PASO1 = ("atencion", "mlp")`.

**Datos**: se usan el índice y la mezcla de fuentes (`config_paso1.json`) de `00_explorar_corpus.ipynb`. Los textos se limpian, se deduplican, se barajan y se empaquetan en bloques de `LARGO_BLOQUE` tokens (`<bos> texto <eos> <bos> texto <eos> ...`).

**Hardware**: A100 40/80 GB recomendada (el modelo ocupa ~16 GB en bf16). En una L4 de 24 GB funciona con lote 1 por GPU (se ajusta solo). Con 5M tokens el entrenamiento toma ~30 min en una A100 (con 20M, ~2 h). Cada checkpoint se sube al repo del Hub (`paso1/<nombre>/last-checkpoint`) y, si la sesión se corta, al volver a ejecutar en otra sesión se baja y se retoma desde ahí.

In [ ]:
%pip install -q -U "transformers>=5.5" "peft>=0.17" accelerate datasets huggingface_hub sacrebleu
%pip uninstall -q -y torchao   # el torchao de Colab (0.10) rompe a peft al crear el LoRA; no se usa

In [ ]:
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
p = torch.cuda.get_device_properties(0)
VRAM_GB = p.total_memory / 1e9
print(f"GPU: {p.name} | {VRAM_GB:.0f} GB | bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Configuración
REPO_HF = "mariopedemonte12/chineutro-gemma4"   # repo privado con datos, checkpoints y resultados; None = solo local
DIR_TRABAJO = None   # None = salidas/gemma4 dentro del repo git, o /content/chineutro en Colab
MODELO_BASE = "google/gemma-4-E4B"   # base pre-entrenado (no la variante -it)
PRESUPUESTO_TOKENS = None            # None = el de config_paso1.json
LARGO_BLOQUE = 1024
MODULOS_PASO1 = ("mlp",)             # solo la FFN; ("atencion", "mlp") para la ablación
LORA_R, LORA_ALPHA, LORA_DROPOUT = 64, 128, 0.05
LR = 1e-4
SECUENCIAS_POR_PASO = 32             # lote efectivo: 32 x 1024 = 32k tokens por paso
FRACCION_VAL = 0.005                 # documentos reservados para medir perplejidad
EVAL_CADA = 50                       # pasos entre evaluaciones y checkpoints (~150 pasos con 5M tokens)
SEED = 42
DTYPE = "bfloat16"                   # "float32" solo para pruebas en GPUs sin bf16 (T4)
MAX_PASOS = -1                       # > 0 corta el entrenamiento en ese paso (pruebas)
# Fracción de los tokens de cada fuente que sale de textos con habla chilena (marcadores o chilenismos del
# diccionario); el resto, al azar. Solo Twitter: en mc4 y news casi no hay y lo que se detecta es ruido.
FRACCION_PRIORITARIA = {"twitter": 0.8}   # {} = muestreo al azar, como antes

### Datos, checkpoints y resultados: repo privado del Hugging Face Hub

Todo vive en el repo privado `REPO_HF`, que es un espejo de `DIR_TRABAJO`: `DIR_TRABAJO/x/y` queda en `x/y` del repo.

| Ruta en el repo | Contenido |
|---|---|
| `datos/` | `comun.py`, `lanzar.py`, los notebooks, `rag_documentos.jsonl`, `dataset_chilenismos_opus.csv`, `split_oficial.csv` |
| `indice_corpus.parquet`, `config_paso1.json`, `muestra_*.parquet`, `presencia_diccionario_*.csv` | salidas de la exploración (00) |
| `paso1/corpus_<M>M.parquet` | corpus del paso 1 ya construido |
| `paso1/<nombre>/last-checkpoint/` | último checkpoint del paso 1 (se reemplaza en cada guardado) |
| `paso1/<nombre>/adaptador/`, `metricas_paso1.json` | adaptador final y métricas |
| `paso2/<condición>/` | adaptador, predicciones y métricas de cada condición |
| `ejecuciones/` | notebook ejecutado y log de cada corrida sin interfaz (`lanzar.py`) |

**Sin interfaz** (runtime de Colab por API): `python datos/lanzar.py <notebook> --repo REPO_HF -p CONDICION paso1+2`. Corre el notebook con papermill y sube el log cada 10 minutos.

**Dentro del repo git** (local) se usan los datos del repo y `DIR_TRABAJO = salidas/gemma4`. Con `REPO_HF = None` no se sube ni se baja nada.

El token de Hugging Face (variable `HF_TOKEN`, `hf auth login` o el secreto `HF_TOKEN` de Colab) necesita **permiso de escritura** para subir resultados, y acceso al corpus restringido.

In [ ]:
# Entorno: token de Hugging Face, comun.py y los datos, y el repo del Hub
import os, sys, json
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

# Sin interfaz no se puede pedir el token: tiene que estar en HF_TOKEN, en el archivo de `hf auth login`
# o en los secretos de Colab
from huggingface_hub import get_token, snapshot_download
if not get_token() and EN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
assert get_token(), "Falta el token de Hugging Face (HF_TOKEN o hf auth login)"


def raiz_repo():
    """Raíz del repo git, si el notebook corre dentro de él."""
    actual = Path.cwd().resolve()
    return next((c for c in [actual, *actual.parents] if (c / "models/gemma4/comun.py").is_file()), None)


RAIZ_REPO = raiz_repo()
if DIR_TRABAJO is None:
    DIR_TRABAJO = RAIZ_REPO / "salidas/gemma4" if RAIZ_REPO else "/content/chineutro"
DIR_TRABAJO = Path(DIR_TRABAJO)
DIR_TRABAJO.mkdir(parents=True, exist_ok=True)

if RAIZ_REPO:   # dentro del repo git: datos y código del repo
    RAIZ, DIR_CODIGO = RAIZ_REPO, RAIZ_REPO / "models/gemma4"
else:           # Colab: comun.py y los datos vienen de datos/ en el repo del Hub
    if REPO_HF:
        snapshot_download(REPO_HF, allow_patterns=["datos/*"], local_dir=DIR_TRABAJO)
    RAIZ = DIR_CODIGO = DIR_TRABAJO / "datos"
    assert (DIR_CODIGO / "comun.py").is_file(), f"No encontré comun.py en {DIR_CODIGO}"
sys.path.insert(0, str(DIR_CODIGO))
import comun

HUB = comun.Hub(REPO_HF, DIR_TRABAJO)   # REPO_HF = None: no sube ni baja nada
for relativa in (comun.RUTA_RAG, comun.RUTA_PARES, comun.RUTA_SPLIT):
    try:
        print("✓", comun.ruta_dato(RAIZ, relativa))
    except FileNotFoundError as e:
        print(f"✗ {e}")
print("Datos:", RAIZ, "| salidas:", DIR_TRABAJO, "| Hub:", REPO_HF or "desactivado")

In [ ]:
# Estilo de gráficos: una serie en azul, grilla y ejes discretos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

AZUL, NARANJA, TINTA, TINTA_2, GRILLA = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.edgecolor": GRILLA, "axes.labelcolor": TINTA_2, "xtick.color": TINTA_2,
                     "ytick.color": TINTA_2, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 10, "axes.titlesize": 11})
pd.set_option("display.max_colwidth", 200)


def barras(serie, titulo, xlabel, log=False, fmt="{:,.0f}"):
    """Una serie por categoría. En escala log se dibujan puntos: el largo de una barra en log engaña."""
    s = serie.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.4 * len(s) + 1.2))
    if log:
        ax.plot(s.values, s.index.astype(str), "o", color=AZUL, markersize=8)
        ax.set_xscale("log")
        ax.grid(axis="y", visible=True)
    else:
        ax.barh(s.index.astype(str), s.values, color=AZUL, height=0.6)
        ax.grid(axis="y", visible=False)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=TINTA_2, fontsize=9)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.set_xlabel(xlabel)
    plt.tight_layout()
    plt.show()


def curva_perdida(historial, titulo, ylabel, guardar=None):
    """Pérdida de entrenamiento (línea) y de validación (puntos) desde trainer.state.log_history."""
    hist = pd.DataFrame(historial)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    if "loss" in hist:
        tr = hist.dropna(subset=["loss"])
        ax.plot(tr.step, tr.loss, color=AZUL, linewidth=2, label="entrenamiento")
    if "eval_loss" in hist:
        ev = hist.dropna(subset=["eval_loss"])
        ax.plot(ev.step, ev.eval_loss, "o", color=NARANJA, markersize=8, markeredgecolor="white", label="validación")
    ax.set_xlabel("paso")
    ax.set_ylabel(ylabel)
    ax.set_title(titulo, loc="left", color=TINTA)
    ax.legend(frameon=False)
    plt.tight_layout()
    if guardar:
        fig.savefig(guardar, dpi=150)
    plt.show()

In [ ]:
# Mezcla de fuentes y filtros que dejó el notebook de exploración
HUB.bajar("config_paso1.json")
with open(DIR_TRABAJO / "config_paso1.json", encoding="utf-8") as f:
    CFG = json.load(f)
PRESUPUESTO_TOKENS = PRESUPUESTO_TOKENS or CFG["presupuesto_tokens"]
print(json.dumps(CFG, indent=2, ensure_ascii=False))

PRIO = "".join(f"_prio{f[:2]}{round(100 * v)}" for f, v in sorted(FRACCION_PRIORITARIA.items()) if v)
NOMBRE = f"{'+'.join(MODULOS_PASO1)}_r{LORA_R}_{PRESUPUESTO_TOKENS // 10**6}M{PRIO}"
DIR_P1 = DIR_TRABAJO / "paso1" / NOMBRE
DIR_P1.mkdir(parents=True, exist_ok=True)
print("Salidas del paso 1:", DIR_P1)

## 1. Construcción del corpus de entrenamiento

In [ ]:
from huggingface_hub import HfFileSystem
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODELO_BASE)
fs = HfFileSystem()
RUTA_CORPUS = DIR_TRABAJO / "paso1" / f"corpus_{PRESUPUESTO_TOKENS // 10**6}M{PRIO}.parquet"
HUB.bajar(RUTA_CORPUS)
if RUTA_CORPUS.exists():
    corpus = pd.read_parquet(RUTA_CORPUS)
else:
    RUTA_INDICE = DIR_TRABAJO / "indice_corpus.parquet"
    HUB.bajar(RUTA_INDICE)
    if RUTA_INDICE.exists():
        indice = pd.read_parquet(RUTA_INDICE)
    else:
        indice = comun.indice_corpus(*comun.listar_parquets(fs=fs))
        indice.to_parquet(RUTA_INDICE)
    priorizador = None
    if PRIO:
        # Qué expresiones del diccionario cuentan como habla chilena: se decide con la muestra de la exploración
        RUTA_MUESTRA = DIR_TRABAJO / f"muestra_{CFG['n_por_fuente_muestra']}.parquet"
        HUB.bajar(RUTA_MUESTRA)
        det = comun.DetectorChilenismos(comun.ruta_dato(RAIZ, comun.RUTA_RAG))
        claves, tabla_claves = comun.claves_coloquiales(pd.read_parquet(RUTA_MUESTRA), det)
        tabla_claves.to_csv(DIR_TRABAJO / "paso1" / "claves_coloquiales.csv", index=False)
        HUB.subir(DIR_TRABAJO / "paso1" / "claves_coloquiales.csv")
        una = tabla_claves[tabla_claves.cuenta & ~tabla_claves.varias_palabras]
        print(f"Expresiones que cuentan: {len(claves)} ({len(una)} de una palabra: {', '.join(una.clave.head(20))}, ...)")
        priorizador = comun.Priorizador(det, claves)
    corpus = comun.construir_corpus_paso1(indice, fs, tok, CFG["pesos"], PRESUPUESTO_TOKENS,
                                          max_caracteres=CFG["max_caracteres"],
                                          min_palabras=CFG["min_palabras"], seed=SEED,
                                          priorizador=priorizador, fraccion_prioritaria=FRACCION_PRIORITARIA)
    corpus.to_parquet(RUTA_CORPUS)
    HUB.subir(RUTA_CORPUS)

if "prioritario" not in corpus:
    corpus["prioritario"] = False
resumen = corpus.groupby(["source", "prioritario"]).n_tokens.agg(textos="count", tokens="sum")
resumen["%_tokens"] = (100 * resumen.tokens / resumen.tokens.sum()).round(1)
display(resumen)
print("Ejemplos prioritarios:")
for t in corpus[corpus.prioritario].text.sample(min(8, int(corpus.prioritario.sum())), random_state=SEED):
    print("  -", t[:160].replace("\n", " "))

In [ ]:
from datasets import Dataset

corpus = corpus.sample(frac=1.0, random_state=SEED).reset_index(drop=True)   # mezcla fuentes dentro de cada bloque
n_val = max(50, int(len(corpus) * FRACCION_VAL))
corpus_val, corpus_tr = corpus.iloc[:n_val], corpus.iloc[n_val:]


def a_bloques(df):
    return Dataset.from_pandas(df[["text"]], preserve_index=False).map(
        lambda b: comun.empaquetar(b, tok, LARGO_BLOQUE), batched=True, batch_size=1000,
        remove_columns=["text"], num_proc=4)


ds_train = a_bloques(corpus_tr)
ds_val = a_bloques(corpus_val)
ds_val = ds_val.select(range(min(200, len(ds_val))))
print(f"train: {len(ds_train):,} bloques = {len(ds_train) * LARGO_BLOQUE / 1e6:.1f}M tokens | val: {len(ds_val)} bloques")
print(tok.decode(ds_train[0]["input_ids"][:120]))

## 2. Modelo y adaptador LoRA

In [ ]:
modelo = comun.cargar_modelo(MODELO_BASE, dtype=DTYPE)
modelo = comun.agregar_lora(modelo, MODULOS_PASO1, LORA_R, LORA_ALPHA, LORA_DROPOUT)

# Verificación: solo hay parámetros entrenables en los módulos pedidos
import re
entrenables = sorted({re.sub(r"\.\d+\.", ".N.", n) for n, p in modelo.named_parameters() if p.requires_grad})
print("\n".join(entrenables))

In [ ]:
# Línea base antes de entrenar (el adaptador recién creado es la identidad: B = 0)
pares = comun.cargar_pares(RAIZ)
val_pares = pares[pares.split == "val"]
textos_val = corpus_val.text.tolist()[:200]


def medir(etapa):
    return {
        "etapa": etapa,
        "ppl_corpus_val": comun.perplejidad(modelo, tok, textos_val, max_len=512, bs=4),
        "ppl_frases_chilenas": comun.perplejidad(modelo, tok, val_pares.frase_chilena.tolist(), bs=32),
        "ppl_frases_neutras": comun.perplejidad(modelo, tok, val_pares.frase_neutralizada.tolist(), bs=32),
    }


antes = medir("base")
print(antes)

## 3. Entrenamiento

Una época sobre el presupuesto de tokens, coseno con 7 % de calentamiento (~10 pasos con 5M tokens). El `LR` sale de `01b_barrido_lr.ipynb`. Cada `EVAL_CADA` pasos se guarda un checkpoint y se sube al Hub. Al empezar se retoma desde el checkpoint local más nuevo o, en una sesión nueva, desde el del Hub.

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

POR_GPU = 8 if VRAM_GB > 70 else 4 if VRAM_GB > 35 else 1   # los logits de 262k de vocabulario pesan
args = TrainingArguments(
    output_dir=str(DIR_P1 / "checkpoints"),
    per_device_train_batch_size=POR_GPU,
    per_device_eval_batch_size=POR_GPU,
    gradient_accumulation_steps=max(1, SECUENCIAS_POR_PASO // POR_GPU),
    num_train_epochs=1,
    max_steps=MAX_PASOS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_steps=0.07,   # ~10 pasos con 5M tokens
    weight_decay=0.0,
    max_grad_norm=1.0,
    bf16=DTYPE == "bfloat16",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_CADA,
    save_strategy="steps",
    save_steps=EVAL_CADA,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)
# Retomar: checkpoint local más nuevo (misma sesión) o el último subido al Hub (sesión nueva), solo si se entrenó
# con este mismo corpus y configuración (huella_corpus.txt). Si algo cambió, se parte de cero.
import shutil
# huella = corpus + configuración: no se retoma un checkpoint de otro corpus ni con otro LR, rango, lote o largo
HUELLA = comun.huella_texto(comun.huella_archivo(RUTA_CORPUS), tuple(MODULOS_PASO1), LORA_R, LORA_ALPHA, LORA_DROPOUT,
                            LR, SECUENCIAS_POR_PASO, LARGO_BLOQUE, MAX_PASOS, SEED)
ULTIMO = DIR_P1 / "last-checkpoint"
locales = sorted((DIR_P1 / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if not locales:
    HUB.bajar(ULTIMO)
candidatos = [str(p) for p in locales[-1:]] + ([str(ULTIMO)] if (ULTIMO / "trainer_state.json").exists() else [])
RETOMAR = next((c for c in candidatos if comun.checkpoint_compatible(c, HUELLA)), None)
for c in candidatos:
    if c != RETOMAR and not RETOMAR:
        print(f"⚠ {c} se entrenó con otro corpus u otra configuración (o no tiene huella): no se retoma")
if not RETOMAR:
    shutil.rmtree(DIR_P1 / "checkpoints", ignore_errors=True)
print("Corpus:", RUTA_CORPUS.name, "| huella:", HUELLA, "| retoma desde:", RETOMAR or "el inicio")

trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=default_data_collator,
                  callbacks=[comun.callback_subir_checkpoints(HUB, HUB.rel(ULTIMO), huella=HUELLA), comun.callback_progreso()])
estado_ck = json.loads((Path(RETOMAR) / "trainer_state.json").read_text()) if RETOMAR else {}
if RETOMAR and estado_ck.get("max_steps", 0) > 0 and estado_ck.get("global_step", 0) >= estado_ck["max_steps"]:
    # El checkpoint ya completó el entrenamiento: retomarlo haría un paso de más. Se cargan sus pesos e historial.
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    from transformers import TrainerState
    set_peft_model_state_dict(modelo, load_file(str(Path(RETOMAR) / "adapter_model.safetensors")))
    trainer.state = TrainerState.load_from_json(str(Path(RETOMAR) / "trainer_state.json"))
    print(f"El checkpoint ya completó el entrenamiento (paso {estado_ck['global_step']}): no se vuelve a entrenar")
else:
    trainer.train(resume_from_checkpoint=RETOMAR)

In [ ]:
DIR_ADAPTADOR = DIR_P1 / "adaptador"
trainer.save_model(str(DIR_ADAPTADOR))
tok.save_pretrained(str(DIR_ADAPTADOR))
HUB.subir(DIR_ADAPTADOR, reemplazar=True)
print("Adaptador del paso 1 guardado en:", DIR_ADAPTADOR)

## 4. Resultados del paso 1

In [ ]:
despues = medir("paso1")
tabla = pd.DataFrame([antes, despues]).set_index("etapa")
tabla["brecha_chilena_vs_neutra"] = tabla.ppl_frases_chilenas / tabla.ppl_frases_neutras
display(tabla.round(2))

with open(DIR_P1 / "metricas_paso1.json", "w", encoding="utf-8") as f:
    json.dump({"config": {"modelo": MODELO_BASE, "modulos": MODULOS_PASO1, "r": LORA_R, "alpha": LORA_ALPHA,
                          "lr": LR, "tokens": PRESUPUESTO_TOKENS, "bloque": LARGO_BLOQUE, "pesos": CFG["pesos"]},
               "perplejidad": tabla.reset_index().to_dict("records"),
               "prioridad": FRACCION_PRIORITARIA,
               "historial": trainer.state.log_history}, f, indent=2, ensure_ascii=False)
HUB.subir(DIR_P1 / "metricas_paso1.json")

In [ ]:
curva_perdida(trainer.state.log_history, "Paso 1: pérdida de NTP sobre el corpus chileno", "pérdida (nats por token)",
              guardar=DIR_P1 / "perdida.png")
HUB.subir(DIR_P1 / "perdida.png")

In [ ]:
# Comparación cualitativa: continuaciones greedy del modelo base y del modelo con el adaptador
INICIOS = ["Oye, ayer fuimos al carrete y", "Weón, no tengo ni una luca, así que",
           "La micro venía repleta y", "Mi polola me dijo que", "Cachai que el profe"]


def continuar(textos, max_new_tokens=40):
    tok.padding_side = "left"
    enc = tok(textos, return_tensors="pt", padding=True).to(modelo.device)
    with torch.no_grad():
        out = modelo.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens, use_cache=True,
                              pad_token_id=tok.pad_token_id)
    return [tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True).strip() for o in out]


modelo.eval()
with modelo.disable_adapter():
    base = continuar(INICIOS)
ajustado = continuar(INICIOS)
for i, b, a in zip(INICIOS, base, ajustado):
    print(f"\n» {i}\n  base:   {b}\n  paso 1: {a}")

**Qué esperar**: la perplejidad sobre el corpus de validación y sobre las frases chilenas del dataset debería bajar más que la de las frases neutras (la brecha chilena/neutra se achica). Si la perplejidad de las frases neutras sube mucho, el modelo está olvidando español general: baja el `LR`, sube el peso de las fuentes formales o reduce el presupuesto.